# PDF Combiner By Month/Year

Upload PDFs with filenames like `Category Name YYYY.MM.pdf`, and this notebook will sort them chronologically and merge them into a single PDF.

In [ ]:
!pip install -q PyPDF2

In [ ]:
from google.colab import files

uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} file(s).")

In [ ]:
import re
from collections import defaultdict
from PyPDF2 import PdfMerger
from google.colab import files

def extract_date(filename):
    """Extract YYYY.MM from filename and return (year, month) for sorting."""
    match = re.search(r'(\d{4})\.(\d{2})\.pdf$', filename, re.IGNORECASE)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return (9999, 99)

def extract_category(filename):
    """Extract category name (everything before the YYYY.MM date)."""
    match = re.match(r'(.+?)\s*\d{4}\.\d{2}\.pdf$', filename, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return "Uncategorized"

# Group files by category
groups = defaultdict(list)
for f in uploaded.keys():
    category = extract_category(f)
    groups[category].append(f)

# Sort each group by date and merge
output_files = []
for category, file_list in sorted(groups.items()):
    file_list.sort(key=extract_date)
    dates = [extract_date(f) for f in file_list if extract_date(f) != (9999, 99)]
    if dates:
        first = f"{dates[0][0]}.{dates[0][1]:02d}"
        last = f"{dates[-1][0]}.{dates[-1][1]:02d}"
        output_name = f"{category} {first}-{last}.pdf"
    else:
        output_name = f"{category}.pdf"

    print(f"\n--- {output_name} ---")
    merger = PdfMerger()
    for f in file_list:
        yr, mo = extract_date(f)
        date_str = f"{yr:04d}.{mo:02d}" if yr != 9999 else "no date"
        print(f"  [{date_str}] {f}")
        merger.append(f)
    merger.write(output_name)
    merger.close()
    output_files.append(output_name)

print(f"\nDone! Created {len(output_files)} merged PDF(s).")
for out in output_files:
    files.download(out)